# Notebook 13 — Condition Analysis Evaluation (Final Corrected 15K)

**Project:** Adaptive Road Vision  
**Project root:** `C:\\Users\\ankit\\Desktop\\Adaptive_Road_Vision`

Evaluate the trained five-class condition classifier from Notebook 12. This notebook does not train or modify the model.

Expected dataset: 15,000 images — 10,500 train, 1,500 validation, 3,000 test. Classes: Day, Night, Rain, Fog, Unknown.

The notebook checks metadata and file paths, loads the trained checkpoint, reports accuracy/balanced accuracy/macro-F1 and per-class metrics, visualizes confusion matrices, inspects low-confidence and misclassified examples, and saves reports under `results/condition/`.

If auto-discovery does not find the final metadata CSV or trained checkpoint, set `METADATA_PATH_OVERRIDE` or `CHECKPOINT_PATH_OVERRIDE` in the first code cell. Run Notebook 12 successfully before running this notebook.

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *list(current.parents[:5])]:
        if (candidate / "datasets").exists() and (candidate / "notebooks").exists():
            return candidate
    fallback = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")
    if fallback.exists():
        return fallback
    return current

PROJECT_ROOT = find_project_root()

# These names and folders match the metadata discovery strategy used in Notebook 12.
DATASET_DIR_CANDIDATES = [
    PROJECT_ROOT / "datasets" / "final_condition_balanced_v2",
    PROJECT_ROOT / "datasets" / "final_condition_balanced",
    PROJECT_ROOT / "datasets" / "final",
]
METADATA_NAMES = [
    "final_dataset_metadata.csv",
    "dataset_metadata.csv",
    "final_metadata.csv",
    "final_dataset.csv",
    "balanced_dataset_metadata.csv",
    "dataset_balance_metadata.csv",
]
METADATA_PATH_OVERRIDE = None
CHECKPOINT_PATH_OVERRIDE = None

metadata_candidates = []
if METADATA_PATH_OVERRIDE:
    metadata_candidates.append(Path(METADATA_PATH_OVERRIDE))
for dataset_dir in DATASET_DIR_CANDIDATES:
    for name in METADATA_NAMES:
        metadata_candidates.extend([
            dataset_dir / "metadata" / name,
            dataset_dir / name,
        ])
metadata_path = next((p for p in metadata_candidates if p.is_file()), None)
if metadata_path is None:
    for dataset_dir in DATASET_DIR_CANDIDATES:
        if dataset_dir.is_dir():
            found = []
            for name in METADATA_NAMES:
                found.extend(dataset_dir.rglob(name))
            if found:
                metadata_path = sorted(found, key=lambda p: (len(p.parts), str(p).lower()))[0]
                break

MODEL_DIR = PROJECT_ROOT / "models" / "condition"
RESULTS_DIR = PROJECT_ROOT / "results" / "condition"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_candidates = []
if CHECKPOINT_PATH_OVERRIDE:
    checkpoint_candidates.append(Path(CHECKPOINT_PATH_OVERRIDE))
checkpoint_candidates.extend([
    MODEL_DIR / "condition_resnet18_best.pth",
    MODEL_DIR / "best_model.pth",
    MODEL_DIR / "best_condition_model.pth",
    MODEL_DIR / "condition_classifier_best.pth",
])
model_path = next((p for p in checkpoint_candidates if p.is_file()), None)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
EXPECTED_TOTAL = 15000
EXPECTED_SPLITS = {"train": 10500, "val": 1500, "test": 3000}
CLASS_NAMES = ["Day", "Night", "Rain", "Fog", "Unknown"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i: name for i, name in enumerate(CLASS_NAMES)}
LOW_CONFIDENCE_THRESHOLD = 0.60

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Metadata:", metadata_path)
print("Checkpoint:", model_path)
print("Results:", RESULTS_DIR)

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}. Edit PROJECT_ROOT.")
if metadata_path is None:
    raise FileNotFoundError(
        "Final metadata CSV was not found. Set METADATA_PATH_OVERRIDE to the exact CSV from Notebook 10.\n"
        "Checked dataset folders:\n" + "\n".join(str(p) for p in DATASET_DIR_CANDIDATES)
    )
if model_path is None:
    raise FileNotFoundError(
        "Trained condition checkpoint not found. Run Notebook 12 first. Expected best checkpoint:\n"
        + str(MODEL_DIR / "condition_resnet18_best.pth")
        + "\nIf saved elsewhere, set CHECKPOINT_PATH_OVERRIDE."
    )

## 1. Load and normalize metadata

This section supports the column names used in the project metadata (`project_condition` / `condition`, `image_path` / `final_image_path`). It stops if required fields are missing or conditions are outside the five supported classes.

In [ ]:
metadata = pd.read_csv(metadata_path)

# Resolve known aliases from the final dataset metadata.
aliases = {
    "project_condition": ["project_condition", "condition", "weather_condition"],
    "final_image_path": ["final_image_path", "image_path", "img_path"],
    "image_id": ["image_id", "source_id", "image_name", "filename"],
    "image_name": ["image_name", "filename", "image_id"],
    "split": ["split", "target_split"],
}
for target, candidates in aliases.items():
    if target not in metadata.columns:
        found = next((c for c in candidates if c in metadata.columns), None)
        if found is not None:
            metadata[target] = metadata[found]

required_cols = ["project_condition", "final_image_path", "image_id", "image_name", "split"]
missing_cols = [c for c in required_cols if c not in metadata.columns]
if missing_cols:
    raise ValueError(
        f"Metadata missing required columns: {missing_cols}. Available columns: {list(metadata.columns)}"
    )

metadata["split"] = metadata["split"].astype(str).str.strip().str.lower()
metadata["project_condition"] = metadata["project_condition"].astype(str).str.strip().str.title()
condition_aliases = {
    "Daytime": "Day", "Day Time": "Day",
    "Nighttime": "Night", "Night Time": "Night",
    "Rainy": "Rain", "Foggy": "Fog",
    "Unknown/Other": "Unknown", "Unk": "Unknown",
}
metadata["project_condition"] = metadata["project_condition"].replace(condition_aliases)

unexpected = sorted(set(metadata["project_condition"].dropna()) - set(CLASS_NAMES))
if unexpected:
    raise ValueError(
        f"Unexpected condition labels: {unexpected}. Expected {CLASS_NAMES}. "
        "Correct the source metadata rather than silently dropping or relabeling rows."
    )
if metadata[required_cols].isna().any().any():
    missing_counts = metadata[required_cols].isna().sum()
    raise ValueError(f"Missing values in required columns:\n{missing_counts[missing_counts > 0]}")

metadata["label"] = metadata["project_condition"].map(CLASS_TO_ID).astype(int)

def parse_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    if isinstance(value, (int, np.integer, float, np.floating)):
        return bool(value)
    text = str(value).strip().lower()
    return text in {"true", "1", "yes", "y", "synthetic", "synthetic_fog"}

if "synthetic" in metadata.columns:
    metadata["synthetic_flag"] = metadata["synthetic"].map(parse_bool)
elif "is_synthetic" in metadata.columns:
    metadata["synthetic_flag"] = metadata["is_synthetic"].map(parse_bool)
elif "record_type" in metadata.columns:
    metadata["synthetic_flag"] = metadata["record_type"].astype(str).str.lower().str.contains("synthetic")
else:
    metadata["synthetic_flag"] = False
    warnings.warn("No synthetic indicator column found; synthetic-specific checks may be incomplete.")

def resolve_image_path(value):
    p = Path(str(value))
    if p.is_file():
        return str(p.resolve())
    candidates = []
    if not p.is_absolute():
        candidates.extend([
            PROJECT_ROOT / p,
            metadata_path.parent / p,
            metadata_path.parent.parent / p,
        ])
    else:
        parts = p.parts
        if "Adaptive_Road_Vision" in parts:
            idx = parts.index("Adaptive_Road_Vision")
            rel_parts = parts[idx+1:]
            candidates.append(PROJECT_ROOT.joinpath(*rel_parts))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    # Preserve a useful path for the missing-file diagnostic.
    return str(candidates[0] if candidates else p)

metadata["final_image_path"] = metadata["final_image_path"].map(resolve_image_path)

print("Metadata rows:", len(metadata))
print("\nSplit counts:")
display(metadata["split"].value_counts().reindex(["train", "val", "test"], fill_value=0).rename_axis("split").to_frame("images"))
print("\nCondition by split:")
display(pd.crosstab(metadata["split"], metadata["project_condition"]).reindex(index=["train", "val", "test"], columns=CLASS_NAMES, fill_value=0))


## 2. Validate dataset size, splits, paths, and leakage

The notebook expects the previously established 15K split. If your final build intentionally differs, update the expected counts only after confirming Notebook 10's audit report.

In [ ]:
actual_split_counts = {str(k): int(v) for k, v in metadata["split"].value_counts().to_dict().items()}
if len(metadata) != EXPECTED_TOTAL:
    raise ValueError(f"Expected {EXPECTED_TOTAL:,} metadata rows, found {len(metadata):,}.")
if actual_split_counts != EXPECTED_SPLITS:
    raise ValueError(f"Split counts differ. Actual={actual_split_counts}; expected={EXPECTED_SPLITS}.")

if metadata["final_image_path"].duplicated().any():
    examples = metadata.loc[metadata["final_image_path"].duplicated(keep=False), "final_image_path"].head(10).tolist()
    raise ValueError(f"Duplicate image paths found. Examples: {examples}")

missing_files = metadata.loc[~metadata["final_image_path"].map(lambda p: Path(p).is_file())]
if len(missing_files):
    missing_csv = RESULTS_DIR / "condition_missing_images.csv"
    missing_files.to_csv(missing_csv, index=False)
    raise FileNotFoundError(f"{len(missing_files)} image files are missing. Details: {missing_csv}")

synthetic_non_train = metadata[metadata["synthetic_flag"] & (metadata["split"] != "train")]
if len(synthetic_non_train):
    raise ValueError(f"Found {len(synthetic_non_train)} synthetic images outside train split.")

source_col = next((c for c in ["source_id", "original_image_id", "source_image_id"] if c in metadata.columns), None)
if source_col:
    original = metadata[~metadata["synthetic_flag"]].copy()
    sets = {s: set(original.loc[original["split"] == s, source_col].astype(str)) for s in ["train", "val", "test"]}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        overlap = sets[a] & sets[b]
        if overlap:
            raise ValueError(f"Possible source-ID leakage between {a} and {b}: {len(overlap)} overlapping IDs.")

print("PASS: 15,000 rows and expected split counts")
print("PASS: all image paths exist")
print("PASS: no duplicate image paths")
print("PASS: synthetic records are training-only")
print("Source-ID leakage check:", "performed" if source_col else "not possible (source ID column absent)")

## 3. Prepare deterministic evaluation data

Evaluation uses the same ImageNet normalization expected by a standard ResNet-18 transfer-learning setup. Do not apply random augmentation to validation or test images.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ConditionEvalDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(row["final_image_path"]) as im:
            image = im.convert("RGB")
        return {
            "image": eval_transform(image),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "image_id": str(row["image_id"]),
            "image_name": str(row["image_name"]),
            "condition": str(row["project_condition"]),
            "synthetic": bool(row["synthetic_flag"]),
            "path": str(row["final_image_path"]),
        }

val_df = metadata.loc[metadata["split"] == "val"].reset_index(drop=True).copy()
test_df = metadata.loc[metadata["split"] == "test"].reset_index(drop=True).copy()

val_loader = DataLoader(
    ConditionEvalDataset(val_df), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    ConditionEvalDataset(test_df), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
print(f"Validation: {len(val_df)} images / {len(val_loader)} batches")
print(f"Test: {len(test_df)} images / {len(test_loader)} batches")


## 4. Load the trained Notebook 12 checkpoint

This step fails if checkpoint weights do not match the expected ResNet-18 architecture. A random/untrained model is never used as a fallback.

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(DEVICE)

try:
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=True)
except TypeError:
    checkpoint = torch.load(model_path, map_location=DEVICE)

if isinstance(checkpoint, dict):
    saved_classes = checkpoint.get("class_names") or checkpoint.get("classes")
    if saved_classes is not None and list(saved_classes) != CLASS_NAMES:
        raise ValueError(
            f"Checkpoint class order {list(saved_classes)} does not match expected order {CLASS_NAMES}. "
            "Update the class mapping only if Notebook 12 was trained with that exact order."
        )
    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint
else:
    raise ValueError("Checkpoint is not a state dictionary or supported checkpoint dictionary.")

if not isinstance(state_dict, dict):
    raise ValueError("Checkpoint does not contain a valid model state dictionary.")

state_dict = {(k[7:] if k.startswith("module.") else k): v for k, v in state_dict.items()}
try:
    model.load_state_dict(state_dict, strict=True)
except RuntimeError as exc:
    raise RuntimeError(
        "Checkpoint weights do not match this ResNet-18, five-class architecture. "
        "Check Notebook 12's architecture, class order, and saved checkpoint."
    ) from exc

model.eval()
print("Loaded trained checkpoint:", model_path)
if isinstance(checkpoint, dict):
    print("Saved epoch:", checkpoint.get("epoch", "not stored"))
    print("Saved validation macro-F1:", checkpoint.get("val_macro_f1", "not stored"))

## 5. Prediction and metrics

Macro-F1 and balanced accuracy are important alongside overall accuracy because condition counts are not perfectly balanced, and Fog/Unknown can be comparatively difficult.

In [ ]:
def make_confusion_matrix(y_true, y_pred, n_classes):
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    for true, pred in zip(y_true, y_pred):
        cm[int(true), int(pred)] += 1
    return cm

def calculate_metrics(y_true, y_pred, n_classes):
    cm = make_confusion_matrix(y_true, y_pred, n_classes)
    total = int(cm.sum())
    accuracy = float(np.trace(cm) / total) if total else 0.0
    precision, recall, f1, support = [], [], [], []
    for i in range(n_classes):
        tp = int(cm[i, i])
        fp = int(cm[:, i].sum() - tp)
        fn = int(cm[i, :].sum() - tp)
        p = tp / (tp + fp) if (tp + fp) else 0.0
        r = tp / (tp + fn) if (tp + fn) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        precision.append(float(p)); recall.append(float(r)); f1.append(float(f))
        support.append(int(cm[i, :].sum()))
    return {
        "accuracy": accuracy,
        "balanced_accuracy": float(np.mean(recall)),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "precision": precision, "recall": recall, "f1": f1, "support": support,
        "confusion_matrix": cm,
    }

def predict_dataset(model, loader):
    model.eval()
    records = []
    all_true, all_pred, all_conf, all_probs = [], [], [], []
    with torch.inference_mode():
        for batch in loader:
            logits = model(batch["image"].to(DEVICE, non_blocking=True))
            probs = torch.softmax(logits, dim=1)
            conf, pred = probs.max(dim=1)
            true = batch["label"].cpu().numpy()
            pred_np = pred.cpu().numpy()
            probs_np = probs.cpu().numpy()
            conf_np = conf.cpu().numpy()
            all_true.extend(true.tolist()); all_pred.extend(pred_np.tolist())
            all_conf.extend(conf_np.tolist()); all_probs.extend(probs_np.tolist())
            for j in range(len(true)):
                records.append({
                    "image_id": batch["image_id"][j],
                    "image_name": batch["image_name"][j],
                    "true_condition": ID_TO_CLASS[int(true[j])],
                    "predicted_condition": ID_TO_CLASS[int(pred_np[j])],
                    "confidence": float(conf_np[j]),
                    "correct": bool(true[j] == pred_np[j]),
                    "synthetic": bool(batch["synthetic"][j]),
                    "image_path": batch["path"][j],
                    **{f"prob_{CLASS_NAMES[k]}": float(probs_np[j, k]) for k in range(len(CLASS_NAMES))}
                })
    return pd.DataFrame(records), np.asarray(all_true), np.asarray(all_pred), np.asarray(all_probs), np.asarray(all_conf)

val_predictions, y_val, p_val, probs_val, conf_val = predict_dataset(model, val_loader)
test_predictions, y_test, p_test, probs_test, conf_test = predict_dataset(model, test_loader)

val_metrics = calculate_metrics(y_val, p_val, len(CLASS_NAMES))
test_metrics = calculate_metrics(y_test, p_test, len(CLASS_NAMES))

summary = pd.DataFrame([
    {"split": "validation", "samples": len(y_val), **{k: val_metrics[k] for k in ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]}},
    {"split": "test", "samples": len(y_test), **{k: test_metrics[k] for k in ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]}},
])
display(summary)


## 6. Per-class metrics and confusion matrices

In [ ]:
def per_class_table(metrics):
    return pd.DataFrame({
        "condition": CLASS_NAMES,
        "support": metrics["support"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
    })

val_per_class = per_class_table(val_metrics)
test_per_class = per_class_table(test_metrics)
print("Validation per-class results"); display(val_per_class)
print("Test per-class results"); display(test_per_class)

test_cm = test_metrics["confusion_matrix"]
raw_cm_df = pd.DataFrame(test_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
raw_cm_df.index.name = "True"; raw_cm_df.columns.name = "Predicted"
row_sums = test_cm.sum(axis=1, keepdims=True)
normalized_cm = np.divide(test_cm, row_sums, out=np.zeros_like(test_cm, dtype=float), where=row_sums != 0)
norm_cm_df = pd.DataFrame(normalized_cm, index=CLASS_NAMES, columns=CLASS_NAMES)
norm_cm_df.index.name = "True"; norm_cm_df.columns.name = "Predicted"

display(raw_cm_df)
display(norm_cm_df.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(test_cm, interpolation="nearest")
ax.set_title("Test condition confusion matrix — raw counts")
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, str(test_cm[i, j]), ha="center", va="center")
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(normalized_cm, interpolation="nearest", vmin=0, vmax=1)
ax.set_title("Test condition confusion matrix — row-normalized")
fig.colorbar(im, ax=ax)
ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j, i, f"{normalized_cm[i, j]:.2f}", ha="center", va="center")
fig.tight_layout(); plt.show()


## 7. Error analysis, confidence, and Fog performance

In [ ]:
test_predictions["error_type"] = np.where(test_predictions["correct"], "correct", "misclassified")
error_analysis = (
    test_predictions.groupby("true_condition")
    .agg(
        samples=("image_id", "size"),
        errors=("correct", lambda x: int((~x).sum())),
        accuracy=("correct", "mean"),
        mean_confidence=("confidence", "mean"),
    )
    .reindex(CLASS_NAMES)
    .rename_axis("true_condition")
    .reset_index()
)
error_analysis["error_rate"] = error_analysis["errors"] / error_analysis["samples"].replace(0, np.nan)
display(error_analysis)

confusion_pairs = []
for true_idx, true_name in enumerate(CLASS_NAMES):
    for pred_idx, pred_name in enumerate(CLASS_NAMES):
        if true_idx != pred_idx and test_cm[true_idx, pred_idx] > 0:
            confusion_pairs.append({
                "true_condition": true_name,
                "predicted_condition": pred_name,
                "count": int(test_cm[true_idx, pred_idx]),
            })
confusion_pairs_df = pd.DataFrame(confusion_pairs, columns=["true_condition", "predicted_condition", "count"])
if len(confusion_pairs_df):
    confusion_pairs_df = confusion_pairs_df.sort_values("count", ascending=False).reset_index(drop=True)
display(confusion_pairs_df.head(20))

low_confidence = test_predictions[test_predictions["confidence"] < LOW_CONFIDENCE_THRESHOLD].sort_values("confidence")
print(f"Low-confidence predictions (< {LOW_CONFIDENCE_THRESHOLD:.2f}): {len(low_confidence)}")
display(low_confidence[["image_id", "true_condition", "predicted_condition", "confidence", "correct", "image_path"]].head(30))

fog_test = test_predictions[test_predictions["true_condition"] == "Fog"].copy()
fog_summary = {
    "samples": int(len(fog_test)),
    "accuracy": float(fog_test["correct"].mean()) if len(fog_test) else None,
    "mean_confidence": float(fog_test["confidence"].mean()) if len(fog_test) else None,
    "recall": float(test_metrics["recall"][CLASS_TO_ID["Fog"]]),
    "f1": float(test_metrics["f1"][CLASS_TO_ID["Fog"]]),
}
print("Fog test summary:", fog_summary)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(test_predictions.loc[test_predictions["correct"], "confidence"], bins=25, alpha=0.7, label="Correct")
ax.hist(test_predictions.loc[~test_predictions["correct"], "confidence"], bins=25, alpha=0.7, label="Incorrect")
ax.axvline(LOW_CONFIDENCE_THRESHOLD, linestyle="--", label=f"Threshold {LOW_CONFIDENCE_THRESHOLD:.2f}")
ax.set_xlabel("Maximum softmax probability")
ax.set_ylabel("Images")
ax.set_title("Test confidence: correct vs incorrect")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Save all evaluation artifacts

Outputs are saved under `results/condition/`. Test predictions are for final reporting; do not use the test set to tune model hyperparameters.

In [ ]:
EVAL_JSON = RESULTS_DIR / "condition_evaluation_report.json"
TEST_PREDICTIONS_CSV = RESULTS_DIR / "condition_test_predictions.csv"
VAL_PREDICTIONS_CSV = RESULTS_DIR / "condition_val_predictions.csv"
ERROR_ANALYSIS_CSV = RESULTS_DIR / "condition_error_analysis.csv"
LOW_CONFIDENCE_CSV = RESULTS_DIR / "condition_low_confidence.csv"
RAW_CM_CSV = RESULTS_DIR / "condition_test_confusion_matrix_raw.csv"
NORM_CM_CSV = RESULTS_DIR / "condition_test_confusion_matrix_normalized.csv"
PER_CLASS_CSV = RESULTS_DIR / "condition_test_per_class_metrics.csv"
SUMMARY_CSV = RESULTS_DIR / "condition_metrics_summary.csv"
CONFUSION_PAIRS_CSV = RESULTS_DIR / "condition_common_confusions.csv"

test_predictions.to_csv(TEST_PREDICTIONS_CSV, index=False)
val_predictions.to_csv(VAL_PREDICTIONS_CSV, index=False)
error_analysis.to_csv(ERROR_ANALYSIS_CSV, index=False)
low_confidence.to_csv(LOW_CONFIDENCE_CSV, index=False)
raw_cm_df.to_csv(RAW_CM_CSV)
norm_cm_df.to_csv(NORM_CM_CSV)
test_per_class.to_csv(PER_CLASS_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)
confusion_pairs_df.to_csv(CONFUSION_PAIRS_CSV, index=False)

report = {
    "project_root": str(PROJECT_ROOT),
    "metadata_path": str(metadata_path),
    "model_path": str(model_path),
    "device": str(DEVICE),
    "image_size": IMAGE_SIZE,
    "expected_dataset": {"total": EXPECTED_TOTAL, "splits": EXPECTED_SPLITS, "classes": CLASS_NAMES},
    "actual_dataset": {
        "total": int(len(metadata)),
        "split_counts": {str(k): int(v) for k, v in metadata["split"].value_counts().to_dict().items()},
        "condition_counts": {str(k): int(v) for k, v in metadata["project_condition"].value_counts().to_dict().items()},
        "synthetic_count": int(metadata["synthetic_flag"].sum()),
    },
    "validation": {
        "samples": int(len(val_df)),
        **{k: float(val_metrics[k]) for k in ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]},
        "per_class": val_per_class.to_dict(orient="records"),
    },
    "test": {
        "samples": int(len(test_df)),
        **{k: float(test_metrics[k]) for k in ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1"]},
        "per_class": test_per_class.to_dict(orient="records"),
    },
    "fog_test": fog_summary,
    "low_confidence": {"threshold": LOW_CONFIDENCE_THRESHOLD, "count": int(len(low_confidence))},
    "outputs": {
        "test_predictions": str(TEST_PREDICTIONS_CSV),
        "validation_predictions": str(VAL_PREDICTIONS_CSV),
        "error_analysis": str(ERROR_ANALYSIS_CSV),
        "low_confidence": str(LOW_CONFIDENCE_CSV),
        "raw_confusion_matrix": str(RAW_CM_CSV),
        "normalized_confusion_matrix": str(NORM_CM_CSV),
        "per_class_metrics": str(PER_CLASS_CSV),
        "metrics_summary": str(SUMMARY_CSV),
        "common_confusions": str(CONFUSION_PAIRS_CSV),
    },
}
with open(EVAL_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print("\nSaved evaluation artifacts to:", RESULTS_DIR)


## 9. How to interpret the results and improve the project

1. **Prioritize macro-F1 and per-class recall**, not accuracy alone. A high overall score can hide weak Fog/Unknown performance.
2. Inspect the normalized confusion matrix. If Night is often predicted as Day, review the misclassified examples and the training labels.
3. If Fog recall is poor, use more *genuine* Fog training examples where possible, carefully verified synthetic Fog examples, and condition-aware sampling in Notebook 12. Do not put synthetic images in validation/test.
4. Tune model architecture, learning rate, class weighting, or augmentations using **validation only**. Re-run this notebook on the test set only after model choices are finalized.
5. Confidence is not calibration. If the deployed system uses confidence thresholds to switch enhancement branches, calibrate thresholds on validation data and evaluate once on test.
6. The condition classifier is only one module. The next step is to connect its Day/Night/ Rain/Fog output to adaptive enhancement, then evaluate whether enhancement improves downstream YOLO detection—not merely whether images look brighter.
7. Report the exact checkpoint, class order, dataset metadata version, metrics, and device for reproducibility.

## Next notebook
**Notebook 14 — Adaptive enhancement integration / training.** Use the condition prediction to route frames to the correct processing branch (minimal processing for Day, low-light enhancement for Night, deraining/visibility restoration for Rain, dehazing for Fog). Keep the original frame available for comparison and evaluate downstream YOLO performance before claiming an improvement.